In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, brier_score_loss, log_loss

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


base_path = "/content/drive/MyDrive/TFG"

train_csv = base_path + "/df_train_clean.csv"
valid_csv = base_path + "/df_valid_clean.csv"

df_train = pd.read_csv(train_csv)
df_valid = pd.read_csv(valid_csv)

print("Train:", len(df_train))
print("Valid:", len(df_valid))

df_train.head()

Train: 12399
Valid: 3091


,Path,patient_id,No Finding,Pneumothorax,Edema,Pleural Effusion,Enlarged Cardiomediastinum,urgent
0,CheXpert-v1.0/train/patient00005/study1/view1_...,5,1.0,NaN,NaN,0.0,NaN,0
1,CheXpert-v1.0/train/patient00009/study1/view1_...,9,NaN,NaN,0.0,NaN,NaN,0
2,CheXpert-v1.0/train/patient00011/study13/view1...,11,NaN,0.0,NaN,0.0,0.0,0
3,CheXpert-v1.0/train/patient00012/study3/view1_...,12,NaN,0.0,NaN,1.0,0.0,1
4,CheXpert-v1.0/train/patient00012/study1/view1_...,12,NaN,NaN,NaN,0.0,0.0,0


In [ ]:
import os

if not os.path.exists("/content/train_clean_images"):
    !unzip -oq "/content/drive/MyDrive/TFG/train_clean_images.zip" -d "/content/"
    print("Train descomprimido")
else:
    print("Train ya estaba descomprimido")

if not os.path.exists("/content/valid_clean_images"):
    !unzip -oq "/content/drive/MyDrive/TFG/valid_clean_images.zip" -d "/content/"
    print("Valid descomprimido")
else:
    print("Valid ya estaba descomprimido")

Train descomprimido
Valid descomprimido


In [ ]:
urgent_cols = [
    "Pneumothorax",
    "Edema",
    "Pleural Effusion",
    "Enlarged Cardiomediastinum"
]

df_train["urgent"] = (df_train[urgent_cols] == 1).any(axis=1).astype(int)
df_valid["urgent"] = (df_valid[urgent_cols] == 1).any(axis=1).astype(int)

print(df_train["urgent"].value_counts())
print(df_valid["urgent"].value_counts())

urgent
0    7455
1    4944
Name: count, dtype: int64
urgent
0    1883
1    1208
Name: count, dtype: int64


In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
class CheXpertDataset(Dataset):
    def __init__(self, dataframe, base_path, transform=None):
        self.df = dataframe
        self.base_path = base_path
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Quitar el prefijo que no existe en la carpeta descomprimida
        relative_path = row["Path"].replace("CheXpert-v1.0/train/", "")
        img_path = os.path.join(self.base_path, relative_path)

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(row["urgent"], dtype=torch.float32)

        return img, label

In [ ]:
print(df_train["Path"].iloc[0])

CheXpert-v1.0/train/patient00005/study1/view1_frontal.jpg


In [ ]:
train_dataset = CheXpertDataset(df_train, "/content/train_clean_images", transform)

img, label = train_dataset[0]

print(img.shape)
print(label)

torch.Size([3, 224, 224])
tensor(0.)


In [ ]:
valid_dataset = CheXpertDataset(df_valid, "/content/valid_clean_images", transform)

img_valid, label_valid = valid_dataset[0]
print(img_valid.shape)
print(label_valid)

torch.Size([3, 224, 224])
tensor(0.)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [ ]:
model = models.resnet50(pretrained=True)
model

In [ ]:
import torch.nn as nn

# Cambiar la última capa
model.fc = nn.Linear(in_features=2048, out_features=1)

model

In [ ]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)

In [ ]:
num_epochs = 10

train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f}")

In [ ]:
model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = torch.sigmoid(outputs)  # convertir a probabilidad
        preds = (probs >= 0.5).float()  # umbral

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)
brier = brier_score_loss(labels, probs)
logloss = log_loss(labels, probs)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("AUC:", auc)
print("Brier Score:", brier)
print("LogLoss:", logloss)

In [ ]:


metricas_df = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1-score", "AUC"],
    "Valor": [accuracy, precision, recall, f1, auc]
})

metricas_df

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/TFG/model_resnet50.pth")
print("Modelo guardado correctamente")

In [ ]:


losses_df = pd.DataFrame({
    "Epoch": list(range(1, len(train_losses)+1)),
    "Loss": train_losses
})

losses_df.to_csv("/content/drive/MyDrive/TFG/train_losses.csv", index=False)
print("Losses guardadas")
losses_df

In [ ]:
metricas_df = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1-score", "AUC"],
    "Valor": [accuracy, precision, recall, f1, auc]
})

metricas_df["Valor"] = metricas_df["Valor"].round(4)

metricas_df.to_csv("/content/drive/MyDrive/TFG/metricas_modelo.csv", index=False)

print("Métricas guardadas")
metricas_df

In [ ]:


np.save("/content/drive/MyDrive/TFG/all_labels.npy", np.array(all_labels))
np.save("/content/drive/MyDrive/TFG/all_preds.npy", np.array(all_preds))
np.save("/content/drive/MyDrive/TFG/all_probs.npy", np.array(all_probs))

print("Predicciones guardadas")

In [ ]:


metricas_df = pd.read_csv("/content/drive/MyDrive/TFG/metricas_modelo.csv")
metricas_df

In [ ]:
losses_df = pd.read_csv("/content/drive/MyDrive/TFG/train_losses.csv")

In [ ]:


all_labels = np.load("/content/drive/MyDrive/TFG/all_labels.npy")
all_preds = np.load("/content/drive/MyDrive/TFG/all_preds.npy")
all_probs = np.load("/content/drive/MyDrive/TFG/all_probs.npy")

In [ ]:


model = models.resnet50(pretrained=False)
model.fc = nn.Linear(2048, 1)

model.load_state_dict(torch.load("/content/drive/MyDrive/TFG/model_resnet50.pth"))
model.eval()

print("Modelo cargado")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(losses_df["Epoch"], losses_df["Loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Evolución de la pérdida durante el entrenamiento")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Matriz de confusión")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss

# Cargar lo ya guardado
all_labels = np.load("/content/drive/MyDrive/TFG/all_labels.npy").ravel()
all_probs = np.load("/content/drive/MyDrive/TFG/all_probs.npy").ravel()

# Cargar la tabla actual
metricas_df = pd.read_csv("/content/drive/MyDrive/TFG/metricas_modelo.csv")

# Calcular nuevas métricas
brier = brier_score_loss(all_labels, all_probs)
logloss = log_loss(all_labels, all_probs)

print("Brier Score:", round(brier, 4))
print("LogLoss:", round(logloss, 4))

# Añadirlas a la tabla
nuevas_metricas = pd.DataFrame({
    "Métrica": ["Brier Score", "LogLoss"],
    "Valor": [round(brier, 4), round(logloss, 4)]
})

metricas_df = pd.concat([metricas_df, nuevas_metricas], ignore_index=True)

# Guardar tabla actualizada
metricas_df.to_csv("/content/drive/MyDrive/TFG/metricas_modelo.csv", index=False)

# Mostrar tabla final
metricas_df

Brier Score: 0.1599
LogLoss: 0.503


,Métrica,Valor
0,Accuracy,0.7807
1,Precision,0.7103
2,Recall,0.7409
3,F1-score,0.7253
4,AUC,0.8353
5,Brier Score,0.1599
6,LogLoss,0.5030
